# Corruption Demo

This notebook shows how to run the corruption pipeline end-to-end: inject known data-quality issues into the clean OCEL SQLite log, then sweep every detector over the result.

Corruption logic lives in `src/order_management_corruption/`. The single entry point is `corrupt_database(src, dst, level=...)` — see the next cell.

## Setup

Before running this notebook, ensure dependencies are installed:
```bash
pip install -e .
```

Or install dependencies directly:
```bash
pip install polars jupyter ipykernel pm4py pydantic pyyaml
```

## Run the corruption

`level` accepts `'legacy'`, `'easy'`, `'medium'`, `'hard'`, or `'all'`. `'all'` runs every injector (24 total across 8 issue types); each tiered level runs one flavor per issue. When `dst_path=None`, the output path defaults to `data/synthetic/order-management-<level>.sqlite` (or `-full.sqlite` for `'all'`).

In [ ]:
import sys, pathlib
ROOT = pathlib.Path.cwd()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.order_management_corruption import corrupt_database, DEFAULT_CLEAN_PATH

DB = corrupt_database(DEFAULT_CLEAN_PATH, level="all")
print(f"Corrupted database: {DB}")

## Full detector sweep

`detect_all` runs every rule-based detector and returns one Polars DataFrame per issue. Use this rather than reimplementing per-issue queries in the notebook.

In [ ]:
from src.detection.error_detection import detect_all

results = detect_all(DB)
print("Detector counts:")
for issue, df in results.items():
    print(f"  {issue:34s} {df.height}")

# Peek at each detector's output — one row per detected violation.
for issue, df in results.items():
    if df.height == 0:
        continue
    print(f"\n=== {issue} ({df.height} rows) ===")
    print(df.head(5))